In [6]:
!/venv/main/bin/pip install transformers peft accelerate torch safetensors bitsandbytes

  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached regex-2026.4.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached transformers-5.5.4-py3-none-any.whl (10.2 MB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
Using cached peft-0.19.1-py3-none-any.whl (680 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (507 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 61.3 MB/s  0:00:01 eta 0:00:01
Using cache

In [2]:
import os
for f in os.listdir("/workspace/mistral-qlora-sentiment-adapter"):
    print(f)

adapter_model.safetensors
README.md
adapter_config.json
.ipynb_checkpoints


In [3]:
from huggingface_hub import snapshot_download
path = snapshot_download(
    repo_id="mistralai/Mistral-7B-v0.1",
    local_dir="/workspace/mistral-7b-base",
    ignore_patterns=["*.bin"],
)
print(path)

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/workspace/mistral-7b-base


In [5]:
import sys
print(sys.executable)

/venv/main/bin/python


In [7]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base_model_path = "/workspace/mistral-7b-base"
adapter_path = "/workspace/mistral-qlora-sentiment-adapter"
output_path = "/workspace/mistral-7b-sentiment-merged"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
model = AutoModelForSequenceClassification.from_pretrained(
    base_model_path,
    num_labels=3,
    quantization_config=bnb_config,
    device_map="cuda",
)
model.config.pad_token_id = tokenizer.eos_token_id

print("Loading adapter...")
model = PeftModel.from_pretrained(model, adapter_path)

print("Merging...")
model = model.merge_and_unload()

print("Saving...")
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

print("Done!")

Loading tokenizer...
Loading base model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/venv/main/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
MistralForSequenceClassification LOAD REPORT from: /workspace/mistral-7b-base
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading adapter...
Merging...


/venv/main/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Saving...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done!


In [8]:
import os
for f in sorted(os.listdir("/workspace/mistral-7b-sentiment-merged")):
    size = os.path.getsize(f"/workspace/mistral-7b-sentiment-merged/{f}")
    print(f"{f:50s} {size/1e9:.2f} GB" if size > 1e8 else f"{f:50s} {size/1e3:.1f} KB")

config.json                                        1.3 KB
model.safetensors                                  3.86 GB
tokenizer.json                                     3505.6 KB
tokenizer_config.json                              0.5 KB


In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = "/workspace/mistral-7b-sentiment-merged"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    device_map="cuda",
)
model.eval()

id2label = {0: "Bearish", 1: "Bullish", 2: "Neutral"}

texts = [
    "Federal Reserve holds interest rates steady",
    "Company reports massive losses this quarter",
    "$AAPL hits all-time high after earnings beat",
]

inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to("cuda")
with torch.no_grad():
    logits = model(**inputs).logits
preds = logits.argmax(dim=-1)

for text, pred in zip(texts, preds):
    print(f"{id2label[pred.item()]:10s} | {text}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Neutral    | Federal Reserve holds interest rates steady
Bearish    | Company reports massive losses this quarter
Bullish    | $AAPL hits all-time high after earnings beat
